<a href="https://colab.research.google.com/github/khiemp110/template/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KoboldCpp / MiniMax H3 Image trên Colab

1. Chọn **GPU runtime**. Bật `UsePreset`, chọn `Template`, rồi chạy cell bên dưới. Tắt `UsePreset` để dùng các checkbox thủ công.
2. `AllowSaveToGoogleDrive = True` mặc định: cấp quyền Drive lần đầu. Model lưu trong **MyDrive/Colab Notebooks/KoboldCPP/models/**. Phiên sau kiểm tra SHA256 rồi dùng lại; tải dở được tiếp tục. Mọi tải model chạy bằng mạng của **máy Colab**, không đi qua máy cá nhân.
3. `DeleteExistingModels` chỉ dọn cache tạm `/content/koboldcpp-cache`. Muốn xóa cache Drive, tự quản lý thư mục trên Drive. Tắt lưu Drive thì cache mất khi runtime bị xóa.
4. **MiniMax H3 Image (ComfyUI)** tự cài ComfyUI ở commit cố định và tải bộ INT8 + text encoder NVFP4 + Image VAE (khoảng **37,8 GB**). Dùng Python 3.12+, GPU và High RAM; không đảm bảo đủ bộ nhớ trên T4/RAM thấp. Mở link, chọn **Workflows → MiniMax H3 Image**, sửa prompt và bấm **Run**. Workflow khởi đầu dùng 1024×1024, Euler, 6 bước, CFG 1. Ảnh lưu tại **Colab Notebooks/KoboldCPP/output/**.
5. Các preset khác chạy KoboldCpp. MiniMax H3 **Image** dùng ComfyUI vì backend KoboldCpp được kiểm tra chưa có Image VAE riêng; hỗ trợ H3 video không thay thế được H3 Image.
6. Có Secret `CLOUDFLARE_API_KEY`: dùng named tunnel đã route tới `http://localhost:5001`. Nếu không, KoboldCpp dùng `--remotetunnel`, ComfyUI dùng Quick Tunnel và in link HTTPS. Bấm **Stop** để dừng phiên và tunnel do cell tạo.

Đọc model nhiều GB từ Drive và kiểm tra SHA256 có thể mất thời gian, nhưng không tải lại trọng số từ Hugging Face. Notebook không tự chạy tạo ảnh trước khi bạn bấm Run trong giao diện.

Nguồn: [KoboldCpp](https://github.com/LostRuins/koboldcpp), [backend MiniMax H3 Image](https://github.com/leejet/stable-diffusion.cpp/blob/master/docs/minimax_h3_image.md), [workflow chính thức](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/image_minimax_h3_image_t2i.json).


In [ ]:
#@title Âm thanh tùy chọn (không đảm bảo giữ runtime) { display-mode: "form" }
%%html
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls></audio>

In [ ]:
# @title <b>v-- Chọn Model bên dưới rồi bấm Play để chạy KoboldCpp nhé peko!</b> { display-mode: "form" }
# @markdown Nhớ bấm nút Play khởi động lại mỗi khi ngươi đổi cấu hình đó!

import hashlib
import json
import os
from pathlib import Path
import shlex
import signal
import shutil
import subprocess
import sys
from urllib.parse import unquote, urlparse
from google.colab import drive, userdata

# @markdown ### 1. Chọn nhanh Preset có sẵn (Tùy chọn)
UsePreset = True # @param {type:"boolean"}
Template = "Qwen Image 2.1 Max (Q8 L4)" # @param ["Gemma4 E4B Uncensored (General)", "Tiefighter 13B (General)", "Estopia 13B (General)", "Qwen3 VL 8B (Vision)", "Cydonia 24B (Roleplay)", "Broken Tutu 24B (Roleplay)", "Dans PersonalityEngine 24B (Roleplay)", "Harbinger 24B (Instruct)", "Muse 12B (Instruct)", "Mistral Small 24B (Corporate)", "GPT OSS 20B (Corporate)", "Gemma3 12B (Corporate)", "GLM4 9b chat (Corporate)", "Z-Image Turbo (Image)", "Flux Klein 4B (Image)", "Qwen Image 2.1 (Image)", "Qwen Image 2.1 Max (Q8 L4)", "MiniMax H3 Image (ComfyUI)"]

# @markdown ### 2. Cấu hình thủ công (chỉ áp dụng khi tắt UsePreset)
# @markdown <hr>
LoadTextModel = False # @param {type:"boolean"}
Model = "Gemma 4 12B (Q4_K_M)" # @param ["Gemma 4 12B (Q4_K_M)", "Gemma 4 E4B Uncensored (Q5_K_M)", "Tiefighter 13B (Q4_K_S)", "Estopia 13B (Q4_K_S)", "Llama 3.1 8B BookAdventures (Q6_K)", "Cydonia 24B v4.2.0 (Q4_K_S)", "Broken Tutu 24B (Q4_K_S)", "Dans PersonalityEngine 24B (Q4_K_S)", "Harbinger 24B (Q4_K_S)", "Muse 12B (Q4_K_S)", "Qwen3 VL 8B Instruct (Q6_K)", "Mistral Small 3.2 24B (Q4_K_S)", "GPT OSS 20B (MXFP4)", "Gemma 3 12B (Q4_K_S)", "Gemma 3n E4B (Q6_K)", "GLM 4 9B (Q6_K)"] {allow-input: true}
Layers = "Auto" # @param ["Auto","999"] {allow-input: true}
ContextSize = "40960" # @param ["4096","8192","12288","16384","24576","32768","40960"] {allow-input: true}

# @markdown <hr>
LoadVisionMMProjector = False # @param {type:"boolean"}
Mmproj = "Gemma 4 E4B mmproj (BF16)" # @param ["Gemma 4 E4B mmproj (BF16)", "Qwen3 VL 8B mmproj (BF16)", "LLaMA 13B mmproj v1.5 (Q4_1)", "LLaMA 3 8B mmproj (Q4_1)"] {allow-input: true}

# @markdown <hr>
LoadImgModel = True # @param {type:"boolean"}
ImgModel = "Qwen Image 2.1 (Q8_0 - Max Quality)" # @param ["Qwen Image 2.1 (Q8_0 - Max Quality)", "Qwen Image 2.1 (Q5_K_M)", "PicX Real (Q8_0)", "DreamShaper XL v2 (Q4_K)", "Flux Klein 4B (Q4_0)", "Flux Klein 9B (Q5_K_M)"] {allow-input: true}
ImgClip1 = "Qwen3 VL 8B Instruct (Q8_0 - Max Text)" # @param ["Qwen3 VL 8B Instruct (Q8_0 - Max Text)", "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)", "Qwen3 4B (Q4_K_M - Low VRAM)"] {allow-input: true}
ImgVAE = "Qwen Image 2.1 VAE (Texture-Fix BF16)" # @param ["Qwen Image 2.1 VAE (Texture-Fix BF16)", "Flux2 VAE"] {allow-input: true}

# @markdown <hr>
LoadSpeechModel = False # @param {type:"boolean"}
SpeechModel = "Whisper Base EN (Q5_1)" # @param ["Whisper Base EN (Q5_1)"] {allow-input: true}

# @markdown <hr>
LoadTTSModel = False # @param {type:"boolean"}
TTSModel = "Kokoro (Q4)" # @param ["Kokoro (Q4)", "OuteTTS 0.2 500M (Q4_0)"] {allow-input: true}
WavTokModel = "None" # @param ["None", "WavTokenizer Large 75 (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadEmbeddingsModel = False # @param {type:"boolean"}
EmbeddingsModel = "Snowflake Arctic Embed S (Q4_0)" # @param ["Snowflake Arctic Embed S (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadMusicModel = False # @param {type:"boolean"}
MusicArgs = "--musiclowvram --musicllm https://huggingface.co/koboldcpp/music/resolve/main/acestep-5Hz-lm-1.7B-BF16.gguf --musicembeddings https://huggingface.co/koboldcpp/music/resolve/main/Ace-Qwen3-Embedding-0.6B-BF16.gguf --musicdiffusion https://huggingface.co/koboldcpp/music/resolve/main/acestep-v15-sftturbo50-Q8_0.gguf --musicvae https://huggingface.co/koboldcpp/music/resolve/main/ace-vae-BF16.gguf"

# @markdown <hr>
# @markdown ### 3. Tiện ích hệ thống & Tunnel
# @markdown DeleteExistingModels chỉ xóa cache tạm /content/koboldcpp-cache, không xóa model trên Drive.
DeleteExistingModels = False # @param {type:"boolean"}
Multiplayer = False # @param {type:"boolean"}
# @markdown Lưu và dùng lại model tại MyDrive/Colab Notebooks/KoboldCPP; Colab tự tải bằng mạng của runtime.
AllowSaveToGoogleDrive = True # @param {type:"boolean"}
MakeLocalTunnelFallback = False # @param {type:"boolean"}

# --- TỪ ĐIỂN ÁNH XẠ TÊN ĐẸP -> LINK HUGGINGFACE ---
MAP_TEXT_MODELS = {
    "Gemma 4 12B (Q4_K_M)": "https://huggingface.co/unsloth/gemma-4-12b-it-GGUF/resolve/main/gemma-4-12b-it-Q4_K_M.gguf",
    "Gemma 4 E4B Uncensored (Q5_K_M)": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
    "Tiefighter 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
    "Estopia 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
    "Llama 3.1 8B BookAdventures (Q6_K)": "https://huggingface.co/KoboldAI/Llama-3.1-8B-BookAdventures-GGUF/resolve/main/Llama-3.1-8B-BookAdventures.Q6_K.gguf",
    "Cydonia 24B v4.2.0 (Q4_K_S)": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf",
    "Broken Tutu 24B (Q4_K_S)": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf",
    "Dans PersonalityEngine 24B (Q4_K_S)": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-Q4_K_S.gguf",
    "Harbinger 24B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf",
    "Muse 12B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf",
    "Qwen3 VL 8B Instruct (Q6_K)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
    "Mistral Small 3.2 24B (Q4_K_S)": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf",
    "GPT OSS 20B (MXFP4)": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf",
    "Gemma 3 12B (Q4_K_S)": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf",
    "Gemma 3n E4B (Q6_K)": "https://huggingface.co/unsloth/gemma-3n-E4B-it-GGUF/resolve/main/gemma-3n-E4B-it-Q6_K.gguf",
    "GLM 4 9B (Q6_K)": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf"
}

MAP_MMPROJ = {
    "Gemma 4 E4B mmproj (BF16)": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf",
    "Qwen3 VL 8B mmproj (BF16)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf",
    "LLaMA 13B mmproj v1.5 (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf",
    "LLaMA 3 8B mmproj (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"
}

MAP_IMG_MODELS = {
    "Qwen Image 2.1 (Q8_0 - Max Quality)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
    "Qwen Image 2.1 (Q5_K_M)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
    "PicX Real (Q8_0)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q8_0.gguf",
    "DreamShaper XL v2 (Q4_K)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/dreamshaperXL_v2_q4_k.gguf",
    "Flux Klein 4B (Q4_0)": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
    "Flux Klein 9B (Q5_K_M)": "https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q5_K_M.gguf"
}

MAP_IMG_CLIPS = {
    "Qwen3 VL 8B Instruct (Q8_0 - Max Text)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
    "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
    "Qwen3 4B (Q4_K_M - Low VRAM)": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf"
}

MAP_IMG_VAES = {
    "Qwen Image 2.1 VAE (Texture-Fix BF16)": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors",
    "Flux2 VAE": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
}

MAP_SPEECH_MODELS = {
    "Whisper Base EN (Q5_1)": "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"
}

MAP_TTS_MODELS = {
    "Kokoro (Q4)": "https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf",
    "OuteTTS 0.2 500M (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf"
}

MAP_WAVTOK_MODELS = {
    "None": "",
    "WavTokenizer Large 75 (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"
}

MAP_EMB_MODELS = {
    "Snowflake Arctic Embed S (Q4_0)": "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"
}

PRESETS = {
    "MiniMax H3 Image (ComfyUI)": {"backend": "comfyui", "model_keys": [
        "minimax-h3-image", "minimax-h3-text-nvfp4", "minimax-h3-image-vae",
    ]},
    "Gemma4 E4B Uncensored (General)": {
        "model": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
        "ctx": "40960",
        "mmproj": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Tiefighter 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Estopia 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Qwen3 VL 8B (Vision)": {
        "model": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
        "ctx": "32768",
        "mmproj": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Cydonia 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf", "ctx": "10240"},
    "Broken Tutu 24B (Roleplay)": {"model": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf", "ctx": "10240"},
    "Dans PersonalityEngine 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-IQ4_XS.gguf", "ctx": "10240"},
    "Harbinger 24B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf", "ctx": "10240"},
    "Muse 12B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf", "ctx": "49152"},
    "Mistral Small 24B (Corporate)": {"model": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf", "ctx": "10240"},
    "GPT OSS 20B (Corporate)": {"model": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf", "ctx": "32768"},
    "Gemma3 12B (Corporate)": {"model": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf", "ctx": "16384"},
    "GLM4 9b chat (Corporate)": {"model": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf", "ctx": "49152"},
    "Z-Image Turbo (Image)": {
        "img_model": "https://huggingface.co/leejet/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_S.gguf",
        "img_vae": "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors"
    },
    "Flux Klein 4B (Image)": {
        "img_model": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf",
        "img_vae": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
    },
    "Qwen Image 2.1 Max (Q8 L4)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    },
    "Qwen Image 2.1 (Image)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    }
}

MINIMAX_MODELS = {'minimax-h3-image': {'repo_id': 'Comfy-Org/MiniMax-H3-Image',
                      'revision': '0bb149f51e3156ed4b90dd45d5620346bfe1df8f',
                      'source_filename': 'diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors',
                      'filename': 'minimax_h3_image_pruned_int8_convrot.safetensors',
                      'folder': 'diffusion_models',
                      'size': 20979314802,
                      'sha256': '83eb140f1ff95c068f8a6886e6d3f3d0d4ab2e07bb2fe6d2ef992a51146ee622'},
 'minimax-h3-text-nvfp4': {'repo_id': 'Comfy-Org/MiniMax-H3',
                           'revision': '54472221d27c695fca9621b0c3ed52f06c4af707',
                           'source_filename': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
                           'filename': 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
                           'folder': 'text_encoders',
                           'size': 15687142551,
                           'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'},
 'minimax-h3-image-vae': {'repo_id': 'Comfy-Org/MiniMax-H3-Image',
                          'revision': '0bb149f51e3156ed4b90dd45d5620346bfe1df8f',
                          'source_filename': 'vae/minimax_h3_image_vae_fp16.safetensors',
                          'filename': 'minimax_h3_image_vae_fp16.safetensors',
                          'folder': 'vae',
                          'size': 1168505312,
                          'sha256': 'd80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af'}}

MINIMAX_WORKFLOW = {'id': 'minimax-h3-image-colab',
 'revision': 0,
 'last_node_id': 9,
 'last_link_id': 11,
 'nodes': [{'id': 1,
            'type': 'UNETLoader',
            'pos': [20, 30],
            'size': [360, 90],
            'flags': {},
            'order': 0,
            'mode': 0,
            'inputs': [],
            'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5], 'slot_index': 0}],
            'properties': {'Node name for S&R': 'UNETLoader'},
            'widgets_values': ['minimax_h3_image_pruned_int8_convrot.safetensors', 'default'],
            'title': 'MiniMax H3 Image / INT8'},
           {'id': 2,
            'type': 'CLIPLoader',
            'pos': [20, 200],
            'size': [360, 115],
            'flags': {},
            'order': 1,
            'mode': 0,
            'inputs': [],
            'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1], 'slot_index': 0}],
            'properties': {'Node name for S&R': 'CLIPLoader'},
            'widgets_values': ['qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
                               'minimax',
                               'default'],
            'title': 'H3 text encoder / NVFP4'},
           {'id': 3,
            'type': 'VAELoader',
            'pos': [20, 400],
            'size': [360, 65],
            'flags': {},
            'order': 2,
            'mode': 0,
            'inputs': [],
            'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [2, 10], 'slot_index': 0}],
            'properties': {'Node name for S&R': 'VAELoader'},
            'widgets_values': ['minimax_h3_image_vae_fp16.safetensors'],
            'title': 'H3 Image VAE / FP16'},
           {'id': 5,
            'type': 'MiniMaxH3ImageConditioning',
            'pos': [460, 180],
            'size': [480, 500],
            'flags': {},
            'order': 3,
            'mode': 0,
            'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1},
                       {'name': 'vae', 'type': 'VAE', 'link': 2}],
            'outputs': [{'name': 'positive',
                         'type': 'CONDITIONING',
                         'links': [4, 6],
                         'slot_index': 0},
                        {'name': 'LATENT', 'type': 'LATENT', 'links': [8], 'slot_index': 1}],
            'properties': {'Node name for S&R': 'MiniMaxH3ImageConditioning'},
            'widgets_values': ['A silver tabby kitten holding a sign reading Hello, natural '
                               'daylight, detailed photography.',
                               1024,
                               1024],
            'title': 'Eye-test house + balloon / 2048 x 2048'},
           {'id': 6,
            'type': 'ConditioningZeroOut',
            'pos': [1020, 550],
            'size': [230, 60],
            'flags': {},
            'order': 4,
            'mode': 0,
            'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 4}],
            'outputs': [{'name': 'CONDITIONING',
                         'type': 'CONDITIONING',
                         'links': [7],
                         'slot_index': 0}],
            'properties': {'Node name for S&R': 'ConditioningZeroOut'},
            'widgets_values': [],
            'title': 'Zero conditioning / CFG-distilled'},
           {'id': 7,
            'type': 'KSampler',
            'pos': [1040, 90],
            'size': [320, 310],
            'flags': {},
            'order': 5,
            'mode': 0,
            'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5},
                       {'name': 'positive', 'type': 'CONDITIONING', 'link': 6},
                       {'name': 'negative', 'type': 'CONDITIONING', 'link': 7},
                       {'name': 'latent_image', 'type': 'LATENT', 'link': 8}],
            'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9], 'slot_index': 0}],
            'properties': {'Node name for S&R': 'KSampler'},
            'widgets_values': [20261112, 'fixed', 6, 1.0, 'euler', 'simple', 1.0],
            'title': 'H3 Image / 6 steps / CFG 1'},
           {'id': 8,
            'type': 'VAEDecode',
            'pos': [1450, 160],
            'size': [230, 70],
            'flags': {},
            'order': 6,
            'mode': 0,
            'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 9},
                       {'name': 'vae', 'type': 'VAE', 'link': 10}],
            'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11], 'slot_index': 0}],
            'properties': {'Node name for S&R': 'VAEDecode'},
            'widgets_values': [],
            'title': 'Decode image'},
           {'id': 9,
            'type': 'SaveImage',
            'pos': [1780, 160],
            'size': [440, 440],
            'flags': {},
            'order': 7,
            'mode': 0,
            'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 11}],
            'outputs': [],
            'properties': {'Node name for S&R': 'SaveImage'},
            'widgets_values': ['MiniMax-H3-Image'],
            'title': 'Save image to Drive'}],
 'links': [[1, 2, 0, 5, 0, 'CLIP'],
           [2, 3, 0, 5, 1, 'VAE'],
           [4, 5, 0, 6, 0, 'CONDITIONING'],
           [5, 1, 0, 7, 0, 'MODEL'],
           [6, 5, 0, 7, 1, 'CONDITIONING'],
           [7, 6, 0, 7, 2, 'CONDITIONING'],
           [8, 5, 1, 7, 3, 'LATENT'],
           [9, 7, 0, 8, 0, 'LATENT'],
           [10, 3, 0, 8, 1, 'VAE'],
           [11, 8, 0, 9, 0, 'IMAGE']],
 'groups': [],
 'config': {},
 'extra': {'ds': {'scale': 0.6, 'offset': [50, 100]}},
 'version': 0.4}

def configure_models(use_preset, template, manual):
    if not use_preset:
        return {**manual, "backend": "koboldcpp"}
    if template not in PRESETS:
        raise ValueError(f"Preset không tồn tại: {template}")
    print(f"Áp dụng preset: {template}")
    return {"backend": "koboldcpp", **PRESETS[template]}


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(url, dest_dir=None, custom_name=None, *, expected_size=None, expected_sha256=None):
    """Runs inside Colab: resume into .part and expose only completed downloads."""
    if not url:
        return None
    parsed = urlparse(url)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError("Model phải là URL HTTPS trực tiếp tới file.")
    filename = custom_name or unquote(Path(parsed.path).name)
    if not filename or Path(filename).name != filename or filename in {".", ".."}:
        raise ValueError("Tên file không hợp lệ.")
    # Distinguish repositories/revisions even when both files are named mmproj.gguf.
    identity = hashlib.sha256(url.encode()).hexdigest()[:16]
    directory = Path(dest_dir or MODEL_CACHE_DIR) / identity
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / filename
    receipt = directory / (filename + ".complete.json")
    partial = directory / (filename + ".part")
    control = directory / (filename + ".part.aria2")

    if target.exists():
        record = json.loads(receipt.read_text()) if receipt.exists() else {}
        known = expected_size is not None and expected_sha256 is not None
        if not known and (record.get("url") != url or not record.get("sha256")):
            raise RuntimeError(f"File chưa có xác nhận tải xong: {target}; kiểm tra hoặc chuyển file đi rồi chạy lại.")
        size = expected_size if known else record["size"]
        digest = expected_sha256 if known else record["sha256"]
        print(f"Kiểm tra cache: {filename} ...", flush=True)
        if target.stat().st_size != size or sha256_file(target) != digest:
            raise RuntimeError(f"Cache sai size/SHA256: {target}; không ghi đè. Chuyển file lỗi đi rồi chạy lại.")
        print(f"Dùng lại model trên {'Drive' if AllowSaveToGoogleDrive else 'runtime'}: {filename}", flush=True)
        return str(target)

    downloaded = partial.stat().st_size if partial.exists() else 0
    if expected_size is not None:
        if downloaded > expected_size:
            raise RuntimeError(f"File tải dở lớn hơn model: {partial}")
        if shutil.disk_usage(directory).free < expected_size - downloaded:
            raise RuntimeError(f"Không đủ dung lượng cho {filename}.")
    print(f"Colab tải → {target} (tiếp tục từ file tải dở nếu có)", flush=True)
    subprocess.run([
        "aria2c", "--console-log-level=warn", "--summary-interval=10",
        "--continue=true", "--auto-file-renaming=false", "--allow-overwrite=true",
        "--file-allocation=none", "--max-tries=5", "--retry-wait=5",
        "--connect-timeout=30", "--timeout=120", "-x", "8", "-s", "8", "-k", "1M",
        "--dir", str(directory), "--out", partial.name, url,
    ], check=True)
    if not partial.is_file() or partial.stat().st_size == 0 or control.exists():
        raise RuntimeError(f"Tải chưa hoàn tất: {partial}. Chạy lại để tiếp tục.")
    if expected_size is not None and partial.stat().st_size != expected_size:
        raise RuntimeError(f"Sai kích thước: {partial}; không sử dụng file này.")
    print(f"Kiểm tra SHA256: {filename} ...", flush=True)
    digest = sha256_file(partial)
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError(f"Sai SHA256: {partial}; chuyển file lỗi đi trước khi tải lại.")
    record = {"url": url, "size": partial.stat().st_size, "sha256": digest}
    target_temporary_receipt = receipt.with_suffix(".json.tmp")
    target_temporary_receipt.write_text(json.dumps(record), encoding="utf-8")
    partial.rename(target)
    target_temporary_receipt.replace(receipt)
    return str(target)


def download_binary(url, target):
    target = Path(target)
    if not target.is_file() or not os.access(target, os.X_OK):
        partial = target.with_suffix(".part")
        subprocess.run(["curl", "--fail", "--location", "--retry", "3",
                        "--output", str(partial), url], check=True)
        partial.chmod(0o755)
        partial.replace(target)
    return str(target)


def prepare_minimax(config):
    if sys.version_info < (3, 12):
        raise RuntimeError("Preset MiniMax H3 Image cần Colab Python 3.12+.")
    print("MiniMax H3 Image dùng ComfyUI; 3 model khoảng 37,8 GB. Chọn GPU + High RAM.", flush=True)
    # Probe in another process; leave notebook's kernel free of CUDA allocations.
    probe = """
import importlib.metadata as m, json, torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trong Colab trước khi chạy.')
print(json.dumps({name:m.version(name) for name in ('torch','torchvision','torchaudio')}))
"""
    packages = json.loads(subprocess.run([sys.executable, "-c", probe], check=True,
                                        capture_output=True, text=True).stdout.strip().splitlines()[-1])
    comfy = Path("/content/ComfyUI")
    ref = "0228a9d73cd7c2daf8f61fe13d56b12efe307171"
    repository = "https://github.com/Comfy-Org/ComfyUI.git"
    if not (comfy / ".git").exists():
        if comfy.exists() and any(comfy.iterdir()):
            raise RuntimeError(f"{comfy} có dữ liệu khác; không ghi đè.")
        comfy.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", str(comfy)], check=True)
        subprocess.run(["git", "remote", "add", "origin", repository], cwd=comfy, check=True)
    origin = subprocess.run(["git", "remote", "get-url", "origin"], cwd=comfy,
                            check=True, capture_output=True, text=True).stdout.strip()
    if origin.removesuffix(".git") not in {repository.removesuffix(".git"), "https://github.com/comfyanonymous/ComfyUI"}:
        raise RuntimeError("ComfyUI có origin khác; không tự thay thế.")
    if subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"], cwd=comfy,
                      check=True, capture_output=True, text=True).stdout.strip():
        raise RuntimeError("ComfyUI có thay đổi chưa lưu; không tự checkout.")
    subprocess.run(["git", "fetch", "--depth", "1", "origin", ref], cwd=comfy, check=True)
    subprocess.run(["git", "checkout", "--detach", ref], cwd=comfy, check=True)
    constraints = Path("/content/koboldcpp-torch-constraints.txt")
    constraints.write_text("".join(f"{name}=={version}\n" for name, version in packages.items()))
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "-r", str(comfy / "requirements.txt"), "-c", str(constraints)], check=True)
    # Reuse the same pinned manifests as models/catalog.json. Downloads happen on Colab.
    workflow = json.loads(json.dumps(MINIMAX_WORKFLOW))
    loader_types = {"diffusion_models": "UNETLoader", "text_encoders": "CLIPLoader", "vae": "VAELoader"}
    required = sum(MINIMAX_MODELS[key]["size"] for key in config["model_keys"])
    missing = 0
    for key in config["model_keys"]:
        model = MINIMAX_MODELS[key]
        url = f"https://huggingface.co/{model['repo_id']}/resolve/{model['revision']}/{model['source_filename']}"
        identity = hashlib.sha256(url.encode()).hexdigest()[:16]
        directory = MODEL_CACHE_DIR / model["folder"] / identity
        if not (directory / model["filename"]).exists():
            missing += max(0, model["size"] - ((directory / (model["filename"] + ".part")).stat().st_size
                          if (directory / (model["filename"] + ".part")).exists() else 0))
    if shutil.disk_usage(MODEL_CACHE_DIR).free < missing:
        raise RuntimeError(f"Cần thêm khoảng {missing / 1e9:.1f} GB trống; toàn bộ preset {required / 1e9:.1f} GB.")
    for key in config["model_keys"]:
        model = MINIMAX_MODELS[key]
        url = f"https://huggingface.co/{model['repo_id']}/resolve/{model['revision']}/{model['source_filename']}"
        path = download_file(url, MODEL_CACHE_DIR / model["folder"],
                             expected_size=model["size"], expected_sha256=model["sha256"])
        relative = Path(path).relative_to(MODEL_CACHE_DIR / model["folder"]).as_posix()
        for node in workflow["nodes"]:
            if node["type"] == loader_types[model["folder"]]:
                node["widgets_values"][0] = relative
    workflows = CACHE_ROOT / "user/default/workflows"
    workflows.mkdir(parents=True, exist_ok=True)
    workflow_path = workflows / "MiniMax H3 Image.json"
    # A separate starter file keeps user edits to the saved workflow intact.
    if not workflow_path.exists():
        workflow_path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2))
    model_config = Path("/content/koboldcpp-comfy-models.yaml")
    model_config.write_text(json.dumps({"koboldcpp_drive": {
        "base_path": str(MODEL_CACHE_DIR), "is_default": True,
        "diffusion_models": "diffusion_models", "text_encoders": "text_encoders", "vae": "vae",
    }}))  # JSON is also valid YAML.
    for name in ("input", "output", "user"):
        (CACHE_ROOT / name).mkdir(parents=True, exist_ok=True)
    print(f"Trong ComfyUI, mở Workflows → MiniMax H3 Image, sửa prompt rồi bấm Run. Ảnh → {CACHE_ROOT / 'output'}", flush=True)
    return [sys.executable, "-u", str(comfy / "main.py"), "--listen", "127.0.0.1", "--port", "5001",
            "--disable-auto-launch", "--disable-all-custom-nodes", "--extra-model-paths-config", str(model_config),
            "--input-directory", str(CACHE_ROOT / "input"), "--output-directory", str(CACHE_ROOT / "output"),
            "--user-directory", str(CACHE_ROOT / "user")], str(comfy)


def build_kobold_command(config):
    binary = download_binary("https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64",
                             "/content/koboldcpp-linux-x64")
    layers = -1 if str(Layers).lower() == "auto" else int(Layers)
    command = [binary, "--gpulayers", str(layers), "--contextsize", str(config.get("ctx", ContextSize)), "--usecublas"]
    for key, flag, mapping in (
        ("model", "--model", MAP_TEXT_MODELS), ("mmproj", "--mmproj", MAP_MMPROJ),
        ("img_model", "--sdmodel", MAP_IMG_MODELS), ("img_clip", "--sdllm", MAP_IMG_CLIPS),
        ("img_vae", "--sdvae", MAP_IMG_VAES), ("speech", "--whispermodel", MAP_SPEECH_MODELS),
        ("tts", "--ttsmodel", MAP_TTS_MODELS), ("wavtok", "--ttswavtokenizer", MAP_WAVTOK_MODELS),
        ("embeddings", "--embeddingsmodel", MAP_EMB_MODELS),
    ):
        value = config.get(key)
        if value:
            url = mapping.get(value, value)
            if url:
                command.extend([flag, download_file(url)])
    if config.get("music"):
        music = shlex.split(config["music"])
        for index, arg in enumerate(music[:-1]):
            if arg in {"--musicllm", "--musicembeddings", "--musicdiffusion", "--musicvae"}:
                music[index + 1] = download_file(music[index + 1])
        command.extend(music)
    if Multiplayer:
        command.append("--multiplayer")
    return command, "/content"


def run_server(command, cwd):
    process = subprocess.Popen(command, cwd=cwd, start_new_session=True)
    try:
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        # Only signal this launcher's process group, including backend descendants.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=5)


manual = {
    "model": Model if LoadTextModel else None,
    "mmproj": Mmproj if LoadVisionMMProjector else None,
    "img_model": ImgModel if LoadImgModel else None,
    "img_clip": ImgClip1 if LoadImgModel else None,
    "img_vae": ImgVAE if LoadImgModel else None,
    "speech": SpeechModel if LoadSpeechModel else None,
    "tts": TTSModel if LoadTTSModel else None,
    "wavtok": WavTokModel if LoadTTSModel else None,
    "embeddings": EmbeddingsModel if LoadEmbeddingsModel else None,
    "music": MusicArgs if LoadMusicModel else None,
}
config = configure_models(UsePreset, Template, manual)
LOCAL_CACHE_ROOT = Path("/content/koboldcpp-cache")
if DeleteExistingModels and LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_CACHE_ROOT)
    print("Đã xóa cache tạm của notebook trong /content. Model trên Drive được giữ lại.")
if AllowSaveToGoogleDrive:
    mount = Path("/content/drive")
    if not (mount.is_mount() and (mount / "MyDrive").is_dir()):
        drive.mount(str(mount))
    if not (mount / "MyDrive").is_dir():
        raise RuntimeError("Drive chưa sẵn sàng; hoàn tất cấp quyền Google rồi chạy lại.")
    CACHE_ROOT = mount / "MyDrive/Colab Notebooks/KoboldCPP"
else:
    CACHE_ROOT = LOCAL_CACHE_ROOT
MODEL_CACHE_DIR = CACHE_ROOT / "models"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cache model: {MODEL_CACHE_DIR}. Tải và đọc model đều do máy Colab thực hiện.")
if shutil.which("aria2c") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)
if config["backend"] == "comfyui":
    cmd_args, working_directory = prepare_minimax(config)
else:
    cmd_args, working_directory = build_kobold_command(config)

cf_token = ""
try:
    cf_token = (userdata.get("CLOUDFLARE_API_KEY") or "").strip()
except Exception:
    pass

# A custom Cloudflare tunnel must route to http://localhost:5001 for either backend.
tunnels = []
try:
    if cf_token or config["backend"] == "comfyui":
        cloudflared = download_binary("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                                     "/content/cloudflared")
        if cf_token:
            print("Mở Cloudflare tunnel đã cấu hình tới localhost:5001.")
            tunnels.append(subprocess.Popen([cloudflared, "tunnel", "run", "--token", cf_token],
                                             stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
        else:
            print("Link ComfyUI https://…trycloudflare.com sẽ xuất hiện trong log Cloudflare bên dưới.", flush=True)
            tunnels.append(subprocess.Popen([cloudflared, "tunnel", "--url", "http://127.0.0.1:5001"]))
    else:
        cmd_args.append("--remotetunnel")
    if MakeLocalTunnelFallback:
        subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)
        tunnels.append(subprocess.Popen(["lt", "--port", "5001"]))
        print("Mật khẩu Localtunnel:", flush=True)
        subprocess.run(["curl", "--fail", "--silent", "https://localtunnel.github.io/bypass"], check=True)
    print(f"\nKhởi động {config['backend']}:\n{shlex.join(cmd_args)}", flush=True)
    run_server(cmd_args, working_directory)
finally:
    cf_token = ""
    for tunnel in tunnels:
        if tunnel.poll() is None:
            tunnel.terminate()
            try:
                tunnel.wait(timeout=5)
            except subprocess.TimeoutExpired:
                tunnel.kill()
                tunnel.wait(timeout=5)
